
### 第7章 引言：地图之外，还能相信吗？

想象一下，你雇了一位极其聪明的向导，他熟记了一片森林里每一条小径的路线。你让他在森林里带路，他几乎从未失手。这就像是人工智能在**训练数据所在的熟悉区域内**的表现——因为这是它“已经见过的区域”，它的每一个判断都有过往的经验作支撑，看起来无比可靠。

但此时此刻，你必须问自己一个严肃的问题：**当你们走出这片熟悉的森林，踏入一片向导从未见过的新区域时，你还能像之前那样信任他吗？**

在这个新区域里，向导可能依然气定神闲地为你指路，但他的指引可能通向绿洲，也可能通向悬崖。在科学计算和机器学习中，这正是我们面临的核心困境：**当AI（或数值模型）给出一个答案时，我们到底凭什么去相信它？**

仔细看这幅引入图，它其实描绘了我们建立“信任”时的三个关键锚点：

第一张图里，在 `[-1, 1]` 的**熟悉训练区域**内，模型的表现曲线是实线，这是有数据支撑的“舒适区”。但一旦我们来到第二张图，把目光投向 `[2, 3]` 这样**没有见过的新区域**时，曲线就变成了虚线和问号。此时，模型给出的答案不再是经验的延续，而是基于某种假设的“外推”。换了一个分布，模型内心其实是一无所知的。

这就引出了第三张图里那个直击灵魂的提问：“**地图外的路，还照旧走吗？**”

科学的态度绝不是盲从，也不是全盘否定，而是明白：**信任，是需要适用条件的。** 如果我们换了环境（分布变了），我们就必须重新寻找证据。

本章要做的核心工作，就是为你提供一套判断“信任边界”的科学工具。我们不讲空泛的哲学，而是要用严密的数学与实验，回答三个极其务实的问题：

1. **在条件理想的情况下，有没有一把标尺能保证模型在没见过的数据上依然可靠？**（我们要用泛化界来丈量风险的底线）
2. **如果环境真的变了，我们该如何设计独立的测试，来诚实地评估模型到底有没有“跨出地图”的能力？**（我们要厘清分布外泛化的真实边界）
3. **就算模型算得又快又好，我们怎么确认它真的解对了问题，且没有在自欺欺人？**

记住，AI的答案之所以可信，从来不是因为它算得足够快、模型足够大，而是因为我们清楚地知道它**是在哪里学的、条件变没变，以及我们能用什么证据去检验它。**

现在，让我们带上这套寻找证据的工具，正式跨出熟悉的边界。


## 7.1 逼近理论与泛化界

在科学计算与机器学习交叉的领域中，我们常常面临两个核心问题：一是我们的模型是否有能力表示真实世界中的复杂规律（**即“能不能表示”**）；二是模型在有限的观测数据上训练后，能否在未知的数据上依然表现良好（**即“能不能推广”**）。前者由逼近理论（Approximation Theory）回答，后者则由泛化理论（Generalization Theory）刻画。本节将深入探讨模型的表达能力与泛化误差界，并分析二者之间的内在张力。

### 7.1.1 逼近理论：模型的表达能力

逼近理论关注的是假设空间 $\mathcal{H}$ 的表达能力——即对于目标函数 $f^*$，是否存在假设 $h \in \mathcal{H}$，使其误差 $\inf_{h\in\mathcal{H}} \|h - f^*\|$（即逼近误差）足够小，这决定了我们选择何种基函数或网络结构来模拟物理场或复杂动力学系统。

**定义 7.1.1（假设类与逼近误差）** 设输入空间为 $\mathcal{X}$，输出空间为 $\mathcal{Y}$，目标函数为 $f:\mathcal{X}\to\mathcal{Y}$。假设类 $\mathcal{H}$ 是若干可测函数 $h:\mathcal{X}\to\mathcal{Y}$ 构成的集合。$\mathcal{H}$ 对 $f$ 的**逼近误差**定义为
$$
\inf_{h\in\mathcal{H}}\ \|h-f\|,
$$
其中 $\|\cdot\|$ 为依据具体问题选定的函数范数（如 $L^2$ 范数或一致范数）。

逼近误差仅由 $\mathcal{H}$ 与 $f$ 的几何关系决定，与样本数量及训练算法无关。它刻画的是假设类的**表示能力**：即使拥有无限数据与完美优化算法，若 $\mathcal{H}$ 中不存在接近 $f$ 的函数，学习过程亦无法产生低误差的预测器。


**定义 7.1.2（万能逼近定理，Universal Approximation Theorem）**  
设 $\mathcal{X}$ 为紧致集。若激活函数 $\sigma$ 是非多项式（如 Sigmoid、ReLU），则对于任意连续函数 $f^* \in C(\mathcal{X})$ 和任意误差精度 $\epsilon > 0$，存在一个单隐层前馈神经网络 $h \in \mathcal{H}$，使得：
$$
\sup_{x \in \mathcal{X}} |h(x) - f^*(x)| < \epsilon
$$
即单隐层神经网络在隐层足够宽时，可以任意精度逼近紧集上的连续函数。这是深度网络“理论上能表示任何东西”的经典依据，也是利用神经网络求解偏微分方程（PINNs）等方法的理论基石。

然而，万能逼近定理只保证了“存在性”，并未保证“效率”。针对某些高维或高振荡函数，浅层网络可能需要极其庞大的参数量。**深度**为此带来了指数级的优势。

**定理 7.1.3（深度的指数级优势）**  
对于某些特定的复合函数（如高振荡的锯齿函数），深度神经网络达到逼近误差 $\epsilon$ 所需的参数规模可能仅为 $O(\text{poly}(\log(1/\epsilon)))$，而浅层网络则需要 $O(\text{poly}(1/\epsilon)))$。这说明在表达效率上，深度架构具有本质优势。

此外，在高维科学计算中，维数灾难是无法回避的痛点。

**例 7.1.1（维数灾难）**  
对一般的 Lipschitz 类函数，逼近误差随输入维数 $d$ 的增长而急剧恶化（如达到 $O(\epsilon^{-d})$ 级），这也是传统核方法、浅层模型在高维情形（如高维相空间模拟）下受限的理论根源。

### 7.1.2 泛化界：从训练误差到测试误差

**定义 7.1.4（泛化误差与经验误差）**  
给定样例集 $D = \{(x_{1},y_{1}),(x_{2},y_{2}),\ldots ,(x_{m},y_{m})\}$，$x_{i}\in \mathcal{X}$，$y_{i}\in \mathcal{Y} = \{- 1, + 1\}$。假设 $\mathcal{X}$ 中的所有样本服从一个隐含未知的分布 $\mathcal{D}$，$D$ 中所有样本都是独立地从这个分布上采样而得，即独立同分布（independent and identically distributed，简称 i.i.d.）样本。

令 $h$ 为从 $\mathcal{X}$ 到 $\mathcal{Y}$ 的一个映射，其泛化误差为

$$
E(h;D) = P_{x\sim \mathcal{D}}(h(x)\neq y), 
$$

$h$ 在 $D$ 上的经验误差为

$$
\widehat{E} (h;D) = \frac{1}{m}\sum_{i = 1}^{m}\mathbb{I}\big(h(x_{i})\neq y_{i}\big). 
$$

由于 $D$ 是 $\mathcal{D}$ 的独立同分布采样，因此 $h$ 的经验误差的期望等于其泛化误差。在上下文明确时，我们将 $E(h;D)$ 和 $\widehat{E} (h;D)$ 分别简记为 $E(h)$ 和 $\widehat{E} (h)$。令 $\epsilon$ 为 $E(h)$ 的上限，即 $E(h)\leqslant \epsilon$；我们通常用 $\epsilon$ 表示预先设定的学得模型所应满足的误差要求，亦称“误差参数”。

虽然假设空间 $\mathcal{H}$ 的表达能力至关重要，但若模型过于复杂，在有限样本下极易发生过拟合。因此，我们需要理论工具来回答：在有限样本下，训练误差与泛化误差之间差距有多大？

本章将研究经验误差与泛化误差之间的逼近程度，若 $h$ 在数据集 $D$ 上的经验误差为 0，则称 $h$ 与 $D$ 一致，否则称其与 $D$ 不一致。对任意两个映射 $h_{1},h_{2}\in \mathcal{X}\to \mathcal{Y}$，可通过其“不合”（disagreement）来度量它们之间的差别：

$$
d(h_{1},h_{2}) = P_{x\sim \mathcal{D}}(h_{1}(x)\neq h_{2}(x)). 
$$

为了给出泛化误差 $L_{\mathcal{D}}(h)$ 与经验误差 $\hat{L}_S(h)$ 之差的上界，即泛化界（Generalization Bound），给出如下定义，经典形式如下：

**定理 7.1.5（一般泛化界形式）**  
在置信度 $1 - \delta$ 下，泛化误差不超过经验误差加上一个由假设空间复杂度和样本量 $n$ 决定的项：
$$
L_{\mathcal{D}}(h) \leqslant \hat{L}_S(h) + \text{复杂度项}(\mathcal{H}, n, \delta)
$$
此定理表明，为了获得良好的泛化能力，我们不仅需要经验误差小，还需要假设空间 $\mathcal{H}$ 的复杂度项足够小。

此外，我们会用到几个常用不等式：

1. Jensen 不等式：对任意凸函数 $f(x)$，有

$$
f(\mathbb{E}(x))\leqslant \mathbb{E}(f(x)). 
$$

2. Hoeffding 不等式 [Hoeffding, 1963]：若 $x_{1}, x_{2}, \ldots , x_{m}$ 为 $m$ 个独立随机变量，且满足 $0 \leqslant x_{i} \leqslant 1$，则对任意 $\epsilon > 0$，有

$$
\begin{array}{l}
P\left(\frac{1}{m}\sum_{i=1}^{m}x_{i}-\frac{1}{m}\sum_{i=1}^{m}\mathbb{E}(x_{i})\geqslant\epsilon\right)\leqslant\exp(-2m\epsilon^{2}), \\
P\left(\left|\frac{1}{m}\sum_{i=1}^{m}x_{i}-\frac{1}{m}\sum_{i=1}^{m}\mathbb{E}(x_{i})\right|\geqslant\epsilon\right)\leqslant2\exp(-2m\epsilon^{2}).
\end{array} 
$$

3. McDiarmid 不等式 [McDiarmid, 1989]：若 $x_{1}, x_{2}, \ldots , x_{m}$ 为 $m$ 个独立随机变量，且对任意 $1 \leqslant i \leqslant m$，函数 $f$ 满足

$$
\sup_{x_{1},\ldots ,x_{m},x_{i}^{\prime}}|f(x_{1},\ldots ,x_{m}) - f(x_{1},\ldots ,x_{i - 1},x_{i}^{\prime},x_{i + 1},\ldots ,x_{m})|\leqslant c_{i},
$$

则对任意 $\epsilon > 0$，有

$$
\begin{array}{l}
P(f(x_{1},\ldots,x_{m})-\mathbb{E}(f(x_{1},\ldots,x_{m}))\geqslant\epsilon)\leqslant\exp\left(\frac{-2\epsilon^{2}}{\sum_{i}c_{i}^{2}}\right), \\
P(|f(x_{1},\ldots,x_{m})-\mathbb{E}(f(x_{1},\ldots,x_{m}))|\geqslant\epsilon)\leqslant2\exp\left(\frac{-2\epsilon^{2}}{\sum_{i}c_{i}^{2}}\right).
\end{array} 
$$

为了具体度量假设空间的复杂度，不同的理论工具被引入（如 VC 维、Rademacher 复杂度等）。我们首先来看针对二分类问题最经典的 VC 维理论。

**定义 7.1.6（VC 维）**  
假设空间 $\mathcal{H}$ 的 VC 维是能被 $\mathcal{H}$ 打散（shatter）的最大示例集的大小。
$$
\mathrm{VC}(\mathcal{H}) = \max \{n : \Pi_{\mathcal{H}}(n) = 2^n\}
$$
其中 $\Pi_{\mathcal{H}}(n)$ 是增长函数。若 $\mathrm{VC}(\mathcal{H}) = d$，则意味着存在大小为 $d$ 的数据集能被 $\mathcal{H}$ 赋予所有可能的标签组合，但不存在大小为 $d+1$ 的数据集能被完全打散。

基于 VC 维，我们可以推导出如下的泛化误差界：

**定理 7.1.7（基于 VC 维的泛化误差界）**  
若假设空间 $\mathcal{H}$ 的 VC 维为 $d$，则对任意 $n > d$，$0 < \delta < 1$ 和 $h \in \mathcal{H}$，以至少 $1 - \delta$ 的概率有：
$$
L_{\mathcal{D}}(h) \leqslant \hat{L}_S(h) + \sqrt{\frac{8d\ln\frac{2en}{d} + 8\ln\frac{4}{\delta}}{n}}
$$
**证明**：  
首先，根据增长函数的 Sauer 引理，有 $\Pi_{\mathcal{H}}(n) \leqslant \left(\frac{e \cdot n}{d}\right)^d$。  
假设空间 $\mathcal{H}$ 中所有假设的泛化误差与经验误差发生大偏差的概率，可以通过联合界（Union Bound）进行控制。对于大小为 $2n$ 的样本集，有：
$$
P\left(\sup_{h \in \mathcal{H}} |L_{\mathcal{D}}(h) - \hat{L}_S(h)| > \epsilon\right) \leqslant 4\Pi_{\mathcal{H}}(2n)\exp\left(-\frac{n\epsilon^2}{8}\right)
$$
我们将 $\Pi_{\mathcal{H}}(2n)$ 的上界代入，得：
$$
4\left(\frac{2en}{d}\right)^d \exp\left(-\frac{n\epsilon^2}{8}\right) \leqslant \delta
$$
对两边取自然对数并整理，得：
$$
d\ln\left(\frac{2en}{d}\right) + \ln\frac{4}{\delta} \leqslant \frac{n\epsilon^2}{8}
$$
解得：
$$
\epsilon \leqslant \sqrt{\frac{8d\ln\frac{2en}{d} + 8\ln\frac{4}{\delta}}{n}}
$$
因此，以至少 \(1 - \delta\) 的概率，泛化误差 $L_{\mathcal{D}}(h) \leqslant \hat{L}_S(h) + \epsilon$ 成立。证毕。

上述 VC 维界体现了“容量越大、样本越少，泛化界越松”的权衡。然而，VC 维是**分布无关**且**数据独立**的，这在面对无限假设空间时往往显得过于保守。Rademacher 复杂度则是一种数据相关的度量方法。

**定义 7.1.8（Rademacher 复杂度）**  
设 $\sigma_i$ 为独立取值的 Rademacher 随机变量（以 0.5 的概率取 \(+1\)，0.5 的概率取 \(-1\)）。函数空间 $\mathcal{F}$ 关于分布 $\mathcal{D}$ 的 Rademacher 复杂度定义为：
$$
R_n(\mathcal{F}) = \mathbb{E}_{Z \sim \mathcal{D}^n}\left[\mathbb{E}_{\sigma}\left[\sup_{f \in \mathcal{F}} \frac{1}{n}\sum_{i=1}^n \sigma_i f(z_i)\right]\right]
$$
它衡量了函数类拟合随机噪声的能力。

**定理 7.1.9（基于 Rademacher 复杂度的泛化界）**  
对假设空间 $\mathcal{H}: \mathcal{X} \to \{-1, +1\}$，以至少 \(1 - \delta\) 的概率有：
$$
L_{\mathcal{D}}(h) \leqslant \hat{L}_S(h) + R_n(\mathcal{H}) + \sqrt{\frac{\ln(1/\delta)}{2n}}
$$
该界不仅与置信度 \(\delta\) 和样本量 \(n\) 有关，还与数据的真实分布 $\mathcal{D}$ 有关，因此通常比基于 VC 维的界更紧致（Tighter）。



无论是基于 VC 维还是 Rademacher 复杂度来推导泛化误差界，所得到的结果均与具体学习算法无关，对所有学习算法都适用。这使得人们能够脱离具体学习算法的设计来考虑学习问题本身的性质，但在另一方面，若希望获得与算法有关的分析结果，则需另辟蹊径。稳定性（stability）分析是这方面一个值得关注的方向。

顾名思义，算法的“稳定性”考察的是算法在输入发生变化时，输出是否会随之发生较大的变化。学习算法的输入是训练集，因此下面我们先定义训练集的两种变化。
损失函数 $\ell (\mathcal{S}_{D}(x),y):\mathcal{Y}\times \mathcal{Y}\to \mathbb{R}^{+}$ 刻画了假设 $\mathcal{S}_{D}$ 的预测标记 $\mathcal{S}_{D}(x)$ 与真实标记 $y$ 之间的差别，简记为 $\ell (\mathcal{S}_{D},z)$。下面定义关于假设 $\mathcal{S}_{D}$ 的几种损失。

*   泛化损失

$$
\ell (\mathcal{S},D) = \mathbb{E}_{x\in \mathcal{X},z = (x,y)}[\ell (\mathcal{S}_{D},z)]. 
$$

*   经验损失

$$
\widehat{\ell} (\mathcal{S},D) = \frac{1}{m}\sum_{i = 1}^{m}\ell (\mathcal{S}_{D},z_{i}). 
$$

*   留一（leave-one-out）损失

$$
\ell_{loo}(\mathcal{S},D) = \frac{1}{m}\sum_{i = 1}^{m}\ell (\mathcal{S}_{D\backslash i},z_{i}).
$$
下面定义算法的均匀稳定性（uniform stability）：

**定义 7.1.10** 对任何 $x\in \mathcal{X},z = (x,y)$，若学习算法 $\mathcal{S}$ 满足

$$
\left|\ell (\mathcal{S}_{D},z) - \ell (\mathcal{S}_{D\backslash i},z)\right|\leqslant \beta ,i = 1,2,\ldots ,m, 
$$

则称 $\mathcal{S}$ 关于损失函数 $\ell$ 满足 $\beta$-均匀稳定性。

显然，若算法 $\mathcal{S}$ 关于损失函数 $\ell$ 满足 $\beta$-均匀稳定性，则有

$$
\begin{aligned}
& \left|\ell (\mathcal{S}_{D},z) - \ell (\mathcal{S}_{D\backslash i},z)\right| \\
& \leqslant \left|\ell (\mathcal{S}_{D},z) - \ell (\mathcal{S}_{D\backslash i},z)\right| + \left|\ell (\mathcal{S}_{D\backslash i},z) - \ell (\mathcal{S}_{D\backslash i},z)\right| \\
& \leqslant 2\beta ,
\end{aligned}
$$

也就是说，移除示例的稳定性包含替换示例的稳定性。

若损失函数 $\ell$ 有界，即对所有 $D$ 和 $z = (x,y)$ 有 $0\leqslant \ell (\mathcal{S}_{D},z)\leqslant M$，则有 [Bousquet and Elisseeff, 2002]：

**定理 7.1.11** 给定从分布 $\mathcal{D}$ 上独立同分布采样得到的大小为 $m$ 的示例集 $D$，若学习算法 $\mathfrak{L}$ 满足关于损失函数 $\ell$ 的 $\beta$-均匀稳定性，且损失函数 $\ell$ 的上界为 $M$，$0< \delta < 1$，则对任意 $m\geqslant 1$，以至少 $1 - \delta$ 的概率有

$$
\begin{array}{l}
\ell (\mathfrak{L},\mathcal{D})\leqslant \widehat{\ell} (\mathfrak{L},D) + 2\beta +(4m\beta +M)\sqrt{\frac{\ln(1 / \delta)}{2m}}, \\
\ell (\mathfrak{L},\mathcal{D})\leqslant \ell_{loo}(\mathfrak{L},D) + \beta +(4m\beta +M)\sqrt{\frac{\ln(1 / \delta)}{2m}}.
\end{array} 
$$

定理给出了基于稳定性分析推导出的学习算法 $\mathfrak{L}$ 学得假设的泛化误差界。经验损失与泛化损失之间差别的收敛率为 $\beta \sqrt{m}$；若 $\beta = O(\frac{1}{m})$，则可保证收敛率为 $O(\frac{1}{\sqrt{m}})$，这与基于 VC 维和 Rademacher 复杂度得到的收敛率一致。


### 7.1.3 两者的张力：近似-估计分解

我们已经看到，模型的表达能力（逼近理论）和泛化能力（泛化界）分别由不同的机制主导。我们可以把总泛化误差拆解为三个部分，从而清晰地看到两者的张力（Trade-off）：

**公式 7.1.12（近似-估计分解）**
$$
\text{总误差} \approx \text{逼近误差} + \text{估计误差} + \text{优化误差}
$$
* **逼近误差**：由容量不足导致。容量太小 $\to$ 逼近误差大（欠拟合）。
* **估计误差**：由容量过大导致。容量太大 $\to$ 估计误差大（过拟合）。
* **优化误差**：由训练不充分导致（如陷入局部最优或未收敛）。

**推论 7.1.13（最优容量区间）**  
逼近理论要求假设空间“要足够大”（以降低逼近误差），而泛化界要求假设空间“不能太大”（以降低估计误差）。二者共同确定了一个最优的容量区间——这就是经典的**偏差-方差权衡（Bias-Variance Trade-off）**的理论形式。

### 7.1.4 深度学习时代的挑战与修正

在本节中，我们深入探讨了科学计算中机器学习模型的理论基础：  
**逼近理论管“能不能表示”，泛化界管“能不能推广”。** 前者依赖函数类与网络深度的表达效率，后者依赖假设空间复杂度与样本量的权衡。两者在**近似-估计分解**中达到了理论上的统一。而当前深度学习的过参数化现象，正推动泛化理论从传统的“容量控制”转向“算法偏好与隐式正则化”的新范式。

在科学计算迈入深度学习时代后，经典的理论框架遇到了前所未有的挑战。在深度神经网络中，参数量动辄以亿计，其 VC 维远大于样本量，按照经典泛化界，界会变得极其松弛（Loose），但网络在实际科学任务中却表现出惊人的泛化能力。这催生了新的理论方向：

1. **基于范数的界**：用权重矩阵的谱范数、Frobenius 范数乘积而非参数量来度量容量（Neyshabur 等），以此解释“过参数化但仍泛化”的现象。
2. **隐式正则化**：Zhang 等人 2017 年的著名实验表明，即使网络能完美拟合随机标签，SGD 依然偏好“简单”解。泛化可能来源于优化算法本身（如 SGD 的隐式偏好），而非仅仅是假设空间的限制。
3. **双下降现象**：Belkin 等 2019 年的工作发现，在过参数化区间，测试误差反而会下降（Double Descent），直接挑战了传统的 U 型泛化曲线。
4. **NTK 与无限宽极限**：Jacot 等通过神经正切核（Neural Tangent Kernel）在核回归视角下重新建立了可分析的泛化保证。

## 7.2 分布内与分布外泛化

7.1 节所建立的泛化界依赖于一个核心前提：训练样本与评价样本来自同一数据分布。一旦评价分布偏离训练分布，Hoeffding 不等式与一致收敛框架所依赖的独立同分布假设即告失效，7.1 节中的界在数学上不再适用。本节的任务是形式化这一偏离，并给出在分布改变条件下仍可使用的误差关系。

### 7.2.1 基本定义

**定义 7.2.1（域与风险）** 设输入空间 $\mathcal{X}$ 与输出空间 $\mathcal{Y}$。一个**域**由联合分布 $P(X,Y)$ 定义，其中 $X\in\mathcal{X}$，$Y\in\mathcal{Y}$。给定假设 $h:\mathcal{X}\to\mathcal{Y}$ 与损失函数 $\ell:\mathcal{Y}\times\mathcal{Y}\to\mathbb{R}_{\ge 0}$，假设 $h$ 在域 $P$ 上的**风险**定义为
$$
R_P(h)=\mathbb{E}_{(X,Y)\sim P}\big[\ell(h(X),Y)\big].
$$
设训练数据来自**源域** $P_S$，评价数据来自**目标域** $P_T$。当 $P_S=P_T$ 时，称评价为**分布内**；当 $P_S\ne P_T$ 时，称评价为**分布外**。

分布外的具体形式中，本节聚焦于**协变量偏移**：输入边际分布发生改变，但给定输入后的条件分布保持不变。这一假设在科学计算任务中具有明确对应：控制方程或物理规律不随参数取值改变，而参数的采样范围在训练与评价阶段不同。

**定义 7.2.2（协变量偏移）** 称源域 $P_S$ 与目标域 $P_T$ 之间发生**协变量偏移**，若
$$
P_S(Y\mid X)=P_T(Y\mid X)\quad \text{对几乎所有 } X,
$$
但边际分布 $P_S(X)\ne P_T(X)$。

在协变量偏移下，目标域风险与源域风险之间的关系完全由输入分布的差异与假设在输入空间上的行为决定。以下引入一个可估计的分布差异度量。

**定义 7.2.3（$\mathcal{H}$-散度）** 设 $\mathcal{H}$ 为取值于 $\{0,1\}$ 的假设类。定义对称差异假设类
$$
\mathcal{H}\Delta\mathcal{H}=\{h\oplus h':h,h'\in\mathcal{H}\},
$$
其中 $\oplus$ 表示异或运算。给定 $\mathcal{X}$ 上的两个分布 $P_X$ 与 $Q_X$，$\mathcal{H}\Delta\mathcal{H}$-**散度**定义为
$$
d_{\mathcal{H}\Delta\mathcal{H}}(P_X,Q_X)=2\sup_{h\in\mathcal{H}\Delta\mathcal{H}}\left|\mathbb{P}_{X\sim P_X}[h(X)=1]-\mathbb{P}_{X\sim Q_X}[h(X)=1]\right|.
$$

该量度量的是假设类 $\mathcal{H}$ 区分两个输入分布的能力。与总变差距离不同，$d_{\mathcal{H}\Delta\mathcal{H}}$ 受假设类复杂度的约束，因而可在有限样本下通过经验估计逼近。

### 7.2.2 协变量偏移下的目标风险界

以下定理给出了协变量偏移下目标域风险与源域风险之间的上界关系。其证明依赖于三角不等式与 $\mathcal{H}\Delta\mathcal{H}$-散度的定义。

**定理 7.2.4（Ben-David 等，2010）** 设 $\mathcal{H}$ 为取值于 $\{0,1\}$ 的假设类。对任意 $h\in\mathcal{H}$，
$$
R_{P_T}(h)\le R_{P_S}(h)+\frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}(P_S(X),P_T(X))+\lambda,
$$
其中
$$
\lambda=\min_{h'\in\mathcal{H}}\big[R_{P_S}(h')+R_{P_T}(h')\big]
$$
称为**理想联合风险**。

**证明** 对任意 $h,h'\in\mathcal{H}$，由三角不等式，
$$
R_{P_T}(h)\le R_{P_T}(h')+R_{P_T}(h,h'),
$$
其中 $R_{P_T}(h,h')=\mathbb{E}_{P_T}[|h(X)-h'(X)|]$ 为 $h$ 与 $h'$ 在目标域上的差异。类似地，
$$
R_{P_S}(h,h')\le R_{P_S}(h)+R_{P_S}(h').
$$
将两式结合，
$$
R_{P_T}(h)\le R_{P_T}(h')+R_{P_S}(h,h')+R_{P_S}(h')-R_{P_S}(h,h')+R_{P_T}(h,h').
$$
对 $h'$ 取最小值，并注意到 $R_{P_T}(h,h')-R_{P_S}(h,h')$ 可被 $d_{\mathcal{H}\Delta\mathcal{H}}(P_S(X),P_T(X))/2$ 上界（因为 $h\oplus h'\in\mathcal{H}\Delta\mathcal{H}$，且 $|h-h'|=h\oplus h'$ 对二值函数成立），即得结论。$\square$

该定理的右端由三项构成：源域风险 $R_{P_S}(h)$、分布差异项 $\frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}$、以及理想联合风险 $\lambda$。源域风险可在训练数据上经验估计；分布差异项可在无标签的源域与目标域输入样本上估计；$\lambda$ 反映的是是否存在一个假设能同时在两个域上表现良好。若不存在这样的假设，则无论分布差异多小，目标域风险都可能很大。

### 7.2.3 重要性加权

协变量偏移下的另一条技术路线是**重要性加权**。其基本思想是对源域样本赋予权重，使加权后的经验风险成为目标域风险的无偏估计。

**定义 7.2.5（重要性权重）** 在协变量偏移假设下，定义**重要性权重**为
$$
w(X)=\frac{P_T(X)}{P_S(X)}.
$$

**命题 7.2.6（重要性加权的无偏性）** 在协变量偏移下，对任意可积函数 $g$，
$$
\mathbb{E}_{P_S}[w(X)g(X,Y)]=\mathbb{E}_{P_T}[g(X,Y)].
$$

**证明** 由条件分布不变性 $P_S(Y\mid X)=P_T(Y\mid X)$，有
$$
\mathbb{E}_{P_S}[w(X)g(X,Y)]=\int w(x)g(x,y)P_S(x,y)\,dx\,dy
=\int \frac{P_T(x)}{P_S(x)}g(x,y)P_S(x)P_S(y\mid x)\,dx\,dy
$$
$$
=\int g(x,y)P_T(x)P_T(y\mid x)\,dx\,dy=\mathbb{E}_{P_T}[g(X,Y)].
$$
$\square$

因此，目标域风险可写为
$$
R_{P_T}(h)=\mathbb{E}_{P_S}[w(X)\ell(h(X),Y)].
$$
然而，重要性加权的实际困难在于 $w(X)$ 通常未知。直接估计两个输入密度再取比值在高维情形下方差极大[citation:3]。Sugiyama 等提出的直接重要性估计方法绕过了密度估计，通过最小化估计密度与真实密度之间的 Kullback-Leibler 散度来学习权重函数[citation:9]。此外，当 $P_S(X)$ 的支撑不能覆盖 $P_T(X)$ 的支撑时，$w(X)$ 在覆盖缺失区域无定义，重要性加权失效。

### 7.2.4 适用边界

定理 7.2.4 的界依赖于若干条件。第一，$\mathcal{H}$ 为二值假设类；对回归问题或一般有界损失，需将 $\mathcal{H}\Delta\mathcal{H}$-散度替换为相应的损失差异度量。第二，$\lambda$ 需要被独立估计或上界，而在实践中 $\lambda$ 本身不可直接计算，通常以源域与目标域上的经验联合风险近似。第三，该界不区分“分布外”的具体机制：若条件分布本身发生改变（概念漂移），协变量偏移假设不成立，定理 7.2.4 的前提即被破坏。

在科学计算语境中，协变量偏移是一种相对乐观的情形：控制方程的形式在训练与评价阶段保持一致，仅参数取值或初值条件发生变化。神经网络算子（如 DeepONet 与 FNO）在同分布样本上可达到极低的预测误差，但在高频或大幅值等分布外输入上误差显著增大[citation:17]。这一实验观察与理论界的含义一致：当目标域输入落在源域输入支撑的低密度区域或支撑之外时，$d_{\mathcal{H}\Delta\mathcal{H}}$ 或重要性权重趋于极端值，界失去实用意义。

### 参考文献

定理 7.2.4 的陈述与证明框架来自 Ben-David、Blitzer、Crammer、Kulesza、Pereira 与 Wortman Vaughan 的 *A theory of learning from different domains*（*Machine Learning*, 79(1–2):151–175, 2010）[citation:2]。该文首次提出了以 $\mathcal{H}\Delta\mathcal{H}$-散度界定目标域风险的方法。Shalev-Shwartz 与 Ben-David 的 *Understanding Machine Learning: From Theory to Algorithms*（Cambridge University Press, 2014）第 25 章对该结果给出了教材化的表述[citation:1]。协变量偏移下重要性加权的系统讨论见 Sugiyama、Krauledat 与 Müller 的 *Covariate shift adaptation by importance weighted cross validation*（*Journal of Machine Learning Research*, 8:985–1005, 2007），直接重要性估计方法见 Sugiyama 等 *Direct importance estimation for covariate shift adaptation*（*Annals of the Institute of Statistical Mathematics*, 60:699–746, 2008）[citation:3][citation:9]。神经算子在分布外样本上的表现数据可见 MDPI *Mathematics* 2026 年第 14 卷的神经算子综述第 6 节与附录 A[citation:17]。

## 7.2.5 实验设计：先预测，再运行

本节实验的目标是将 7.2.2 节中协变量偏移的理论关系置于可控的多项式回归问题中加以观察，并在此基础上给出针对参数化 PDE 任务的分布外划分方案。实验的核心设计原则是：拟合区间与评价区间必须由输入变量显式区分，而非在同一区间内随机划分样本点。多项式回归在外推区间的行为已有系统的数值研究可供参照：三次样条在插值区间内误差为零，但在外推区间极不稳定；高阶多项式在插值中精度较高，外推时仍受过拟合影响而边缘失稳[citation:1]。

**实验模型与数据生成。** 取一维输入 $x$，真实目标函数 $f(x)=\sin(x)$。观测标签 $y=f(x)+\eta$，其中 $\eta\sim\mathcal{N}(0,\sigma^2)$，$\sigma=0.1$。假设类取三次多项式回归与五次多项式回归，以最小二乘拟合。训练区间取 $[0,4]$，评价分三个区间：**内插区** $[0.5,3.5]$（训练区间内部）、**邻近外推区** $[4.5,6.0]$（紧邻训练边界之外）、**远距外推区** $[7.0,9.0]$（远离训练覆盖）。每个评价区间取 $N=200$ 个等距测试点，训练样本取 $m=50$ 个在 $[0,4]$ 上均匀采样的带噪观测。重复 $T=100$ 次独立采样，报告各区间均方根误差的均值与标准差。

针对 PDE 任务的划分方案，以参数化热方程 $\partial_t u=\alpha\partial_{xx}u$ 为例。参数 $\alpha$ 在训练集取 $[0.1,0.5]$，验证集取 $[0.1,0.5]$ 内随机子集用于选模，测试集分两档：**分布内测试** $\alpha\in[0.1,0.5]$（与训练同区间但独立采样），**分布外测试** $\alpha\in[0.6,1.0]$（超出训练范围）。初值条件在训练与测试中均取同一函数族（如 $u(x,0)=\sin(k\pi x)$，$k\in\{1,2\}$），以隔离参数外推这一单一变量。PDE 方案的实验在本节中仅作为设计示例，不要求实际运行。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 回答：多项式回归在内插区、邻近外推区与远距外推区的误差如何变化；五次多项式是否比三次多项式在外推中更不稳定。预测：内插区误差接近噪声水平；邻近外推区误差上升但尚可辨识；远距外推区误差急剧增大，五次多项式的发散程度高于三次多项式。 |
| 输入、参数、单位、数据来源与划分 | 输入 $x$ 无量纲；$f(x)=\sin(x)$；噪声 $\sigma=0.1$；训练区间 $[0,4]$，$m=50$；内插评价 $[0.5,3.5]$，$N=200$；邻近外推 $[4.5,6.0]$，$N=200$；远距外推 $[7.0,9.0]$，$N=200$；重复 $T=100$；伪随机种子固定。PDE 方案参数范围见上文。 |
| 独立参考解/基线及选择理由 | 参考解为 $\sin(x)$ 本身，因其解析已知，可精确计算任意评价点的真值。基线为训练区间上的常数预测 $h_0(x)=\bar{y}_{\text{train}}$，其误差在三个区间上均可解析计算，用于指示“无外推能力”时的误差尺度。 |
| 误差指标、容差及其依据 | 各评价区间的均方根误差 RMSE$=\sqrt{\frac{1}{N}\sum_{i=1}^N(\hat{y}_i-f(x_i))^2}$。容差取 $0.05$，依据为噪声标准差 $0.1$ 的一半，用于区分“接近噪声水平”与“显著超出”。报告 $T=100$ 次重复的均值与标准差。 |
| 环境、种子、设备与运行预算 | Python 3.11，NumPy 1.26，scikit-learn 1.4；种子 2026；单核 CPU；预计运行时间小于 2 分钟。PDE 方案不运行。 |

实验的变量关系可形式化如下。设训练区间为 $\mathcal{X}_{\text{train}}=[0,4]$，评价区间为 $\mathcal{X}_{\text{eval}}\subset\mathbb{R}$。当 $\mathcal{X}_{\text{eval}}\subseteq\mathcal{X}_{\text{train}}$ 时，多项式回归在最小二乘意义下以参数速率收敛，误差随 $m$ 增大而减小；当 $\mathcal{X}_{\text{eval}}\cap\mathcal{X}_{\text{train}}=\emptyset$ 时，误差不再由训练样本密度控制，而由多项式在区间外的增长行为决定。对 $d$ 次多项式，其外推误差的增长阶大致随 $|x-x_{\text{train}}|^d$ 放大，因此更高阶多项式在远距外推中通常更不稳定，但这不构成一般性的泛化界——本实验仅展示特定函数族与特定模型族下的经验现象。

## 7.2.6 结果解释与本节结论

**预期结果的分析框架。** 若实验按上述参数运行，各区间误差应呈现以下模式。第一，内插区 $[0.5,3.5]$ 的 RMSE 对两种多项式均应接近或略高于噪声水平 $0.1$，三次与五次之间的差异不显著。第二，邻近外推区 $[4.5,6.0]$ 的 RMSE 应明显上升，五次多项式可能在此区间已出现边缘振荡，其 RMSE 的方差大于三次多项式。第三，远距外推区 $[7.0,9.0]$ 的 RMSE 对五次多项式应显著大于三次多项式，且随外推距离增大而急剧增大。这一模式与已有数值研究中“三次样条外推极不稳定、高阶多项式在外推边缘失稳”的观察一致[citation:1][citation:6]。

**本节结论。** 分布内评价与分布外评价的区分不在于误差数值的大小，而在于评价样本的输入分布是否独立于训练分布。多项式外推实验表明，当评价区间超出训练区间时，误差不再受训练样本密度控制，且模型阶数对误差增长的影响依赖于具体函数。定理 7.2.4 给出了协变量偏移下目标风险的界，但其分布差异项与外推距离正相关，在外推区域迅速增大至无信息量。因此，分布外泛化的可靠评估要求显式构造独立于训练覆盖范围的测试条件，而非在训练分布内部做随机划分。

**随堂提问：设计含数据泄漏的划分方案，让读者修正并解释。**

问题条件：给定一个参数化 ODE 数据集，参数 $\theta\in[0,1]$，每个 $\theta$ 对应一条时间序列 $u(t;\theta)$，$t\in[0,10]$ 离散为 $100$ 个点。共 $50$ 个参数取值，总计 $5000$ 个数据点。方案 A：随机将 $5000$ 个点按 $80\%/20\%$ 拆分为训练与测试。方案 B：按参数 $\theta$ 分组，取 $40$ 个 $\theta$ 的时间序列为训练，$10$ 个 $\theta$ 的时间序列为测试。方案 C：在方案 B 基础上，测试参数取 $\theta\in[0.8,1.0]$，训练参数取 $\theta\in[0,0.7]$。

同学可能的回答与辨析。对方案 A，同学可能认为随机拆分是最中性的做法。辨析：同一 $\theta$ 下的时间点高度相关，随机拆分使训练与测试共享 $\theta$，测试误差反映时间插值而非参数泛化，构成数据泄漏。对方案 B，同学可能认为按参数分组已足够。辨析：方案 B 消除了同参数泄漏，但若 $50$ 个 $\theta$ 在 $[0,1]$ 上均匀采样，训练与测试参数的取值可能交错分布，测试参数可能紧邻训练参数，属于内插式跨参数评价。对方案 C，同学可能认为外推划分过于严格。辨析：方案 C 是唯一构造了**参数外推**的方案：测试参数区间 $[0.8,1.0]$ 与训练区间 $[0,0.7]$ 不重叠，此时评价的是模型对超出训练覆盖范围的参数取值的预测能力。代价是测试参数数量可能不足（取决于 $\theta$ 的采样密度），且外推误差可能很大，但这正是分布外评价需要诚实报告的结果。

本组最终解释：划分方案的合法性取决于所声称的评价目标。若目标是“参数内插”，方案 B 可接受，但需在报告中注明测试参数位于训练参数范围之内。若目标是“参数外推”，则必须采用方案 C 式的区间不相交设计，且测试参数区间的选择需独立于模型表现（不能根据测试误差反复调整区间边界）。数据泄漏的本质是训练与测试共享了评价目标所不应共享的结构信息；在参数化问题中，最常被共享的结构信息正是参数取值本身。

### 参考文献

多项式回归与三次样条在外推中的数值对比见 Puiu 与 Puiu 的 *Comparative Analysis of Interpolation and Extrapolation Methods*（*Scientific Bulletin of the Politehnica University of Bucharest*, 2024），该文报告了样条外推的不稳定性与高阶多项式外推的边缘失稳[citation:1]。分布偏移下目标风险的理论框架见 Ben-David 等 *A theory of learning from different domains*（*Machine Learning*, 79:151–175, 2010）[citation:8]。协变量偏移与重要性加权的系统讨论见 Sugiyama、Krauledat 与 Müller 的 *Covariate shift adaptation by importance weighted cross validation*（*Journal of Machine Learning Research*, 8:985–1005, 2007）[citation:4]。神经算子在分布外频率条件下的性能退化见变系数波动方程的神经算子研究（arXiv:2605.12997）[citation:2]。数据泄漏与随机拆分低估测试误差的系统讨论见化学机器学习中的泄漏机制分析（*GCB Bioenergy*, 2025）[citation:5]与钙钛矿光伏的 provenance-grouped 评价协议（arXiv:2610.12766）[citation:15]。

## 7.3 机器学习与降阶模型的共性：误差界与 V&V

7.1 节与 7.2 节讨论的是统计意义下的泛化风险与分布偏移。然而，一个更为基础的问题贯穿于机器学习与科学计算两类方法之中：**所构造的近似模型是否在其声称的适用范围内具有可验证的精度？** 机器学习中的神经网络代理模型与科学计算中的降阶模型（Reduced-Order Model, ROM）在形式上分属不同学科传统，但在误差来源的结构、可信性评估的逻辑以及适用边界的刻画方式上具有深刻的共性。本节以这一共性为线索，建立统一的误差分析框架，并说明验证与确认（V&V）如何为两类模型提供可操作的可信性判据。

### 7.3.1 基本定义

**定义 7.3.1（降阶模型）** 设全阶模型（Full-Order Model, FOM）由参数化方程或高维离散系统描述，其状态维度为 $N$。**降阶模型**是一个映射 $\hat{\mathcal{M}}: \mathcal{P} \to \mathcal{V}_r$，其中 $\mathcal{P}$ 为参数空间，$\mathcal{V}_r$ 为维度 $r \ll N$ 的约化空间。ROM 的目标是在参数 $\mu \in \mathcal{P}$ 下以远低于 FOM 的计算代价给出状态或输出的近似。

ROM 的构造通常遵循两条路径。**投影型降阶**通过选取约化基 $\Phi \in \mathbb{R}^{N \times r}$ 将状态约束于低维子空间，并 Galerkin 投影全阶方程得到约化动力学。**非侵入型降阶**则直接利用 FOM 的输入-输出快照数据拟合降阶算子或映射，与机器学习代理模型在形式上趋同。当投影型 ROM 的约化算子由数据通过最小二乘推断而非投影获得时，该方法即 Operator Inference，其与机器学习的边界已不再清晰。

**定义 7.3.2（代理模型误差的分解）** 设 $u(\mu)$ 为 FOM 在参数 $\mu$ 下的精确解，$\hat{u}_r(\mu)$ 为 ROM 输出。误差 $e(\mu) = \|u(\mu) - \hat{u}_r(\mu)\|$ 通常可分解为
$$
e(\mu) \le \underbrace{\inf_{v \in \mathcal{V}_r} \|u(\mu) - v\|}_{\text{投影/表示误差}} + \underbrace{\|\hat{u}_r(\mu) - \mathcal{P}_{\mathcal{V}_r} u(\mu)\|}_{\text{模型/回归误差}},
$$
其中 $\mathcal{P}_{\mathcal{V}_r}$ 为到约化空间的投影。第一项由约化基的表示能力决定，与 7.1 节中的逼近误差对应：即使拥有完美数据与优化算法，若 $\mathcal{V}_r$ 不包含接近 $u(\mu)$ 的函数，误差无法降低。第二项由降阶算子或回归映射的估计精度决定，在机器学习语境中对应泛化误差，在 ROM 语境中对应算子推断误差或投影误差。

**定义 7.3.3（Verification 与 Validation 在代理模型语境下的含义）** 在科学计算 V&V 框架中，**Verification** 确认计算实现是否正确求解了所写的数学模型；**Validation** 确认数学模型是否适合目标用途。对代理模型（包括机器学习代理与 ROM），Verification 扩展为确认代理模型在其训练或构造数据所覆盖的参数范围内，是否以预期的精度逼近 FOM 或参考解；Validation 则要求代理模型的输出与独立的可信观测或更高保真度参考进行对比，且对比条件必须超出构造数据的覆盖范围。

这一扩展的关键在于：代理模型的“适用性”不是全局属性，而是参数空间中的局部属性。Blonigan 与 Parish 的研究表明，双加权残差（DWR）方法能够估计投影型 ROM 在其训练集之外的参数区域中的误差，而机器学习误差模型（MLEM）更适合于 ROM 训练集内部的插值场景。这一发现直接对应 7.2 节的核心结论：分布内与分布外的误差行为具有本质差异，误差估计方法本身也必须考虑适用区域。

### 7.3.2 误差来源的共性结构

机器学习代理模型与 ROM 的误差在现象上表现不同，但其来源可归入三个共同的类别。

**第一，表示误差。** 机器学习中体现为假设类（网络架构与宽度）的表达能力限制；ROM 中体现为约化基维度 $r$ 对解流形的覆盖能力。二者均与数据量无关。在 ROM 中，POD 基的截断误差给出了表示误差的显式上界；在机器学习中，万能逼近定理给出的是存在性保证，而非可计算的有限样本界。这一差异解释了为何 ROM 的数学分析通常更为直接：约化基的构造是确定性的，而神经网络的训练是随机的。

**第二，估计/拟合误差。** 机器学习中，从有限训练样本估计网络参数引入统计波动；ROM 中，从快照数据推断约化算子（如 Operator Inference 中的最小二乘问题）同样引入估计不确定性。二者均随样本量（或快照数）增加而减小，但受数据覆盖范围的制约。Operator Inference 的一个已知性质是：在状态轨迹数据充分且方程结构线性的条件下，推断的算子以 Frobenius 范数收敛于侵入式投影得到的约化算子。

**第三，分布/参数偏移误差。** 机器学习代理在训练分布之外的输入上性能退化；ROM 在参数范围之外的外推行为同样不可靠。对高速翼型流场的神经网络降阶模型的外推测试表明，当马赫数从训练集范围逐步增大时，均方根误差从 $0.0031$ 单调上升至 $0.0178$，当马赫数超过 $13$ 时模型预测精度已不满足要求。这一现象与 7.2 节中协变量偏移的理论分析一致：目标域输入落入源域支撑之外时，任何基于同分布假设的误差界均失效。

**定理 7.3.4（代理模型误差的分解界）** 设 $\mathcal{V}_r$ 为约化空间，$\hat{u}_r$ 为任一构造方法给出的 ROM 或代理模型输出。则对任意 $\mu \in \mathcal{P}$，
$$
\|u(\mu) - \hat{u}_r(\mu)\| \le \inf_{v \in \mathcal{V}_r} \|u(\mu) - v\| + \|\hat{u}_r(\mu) - \mathcal{P}_{\mathcal{V}_r} u(\mu)\|.
$$

**证明** 由三角不等式，
$$
\|u - \hat{u}_r\| = \|u - \mathcal{P}_{\mathcal{V}_r}u + \mathcal{P}_{\mathcal{V}_r}u - \hat{u}_r\| \le \|u - \mathcal{P}_{\mathcal{V}_r}u\| + \|\mathcal{P}_{\mathcal{V}_r}u - \hat{u}_r\|.
$$
由投影的最优性，$\|u - \mathcal{P}_{\mathcal{V}_r}u\| = \inf_{v\in\mathcal{V}_r}\|u - v\|$。$\square$

该分解的意义在于将“模型不够好”与“数据不够多”两个来源分离。表示误差由架构或基的选择决定，在训练开始前即已确定上界；估计误差可通过更多数据或更有效的优化减小，但受表示误差的下限约束。二者共同界定了代理模型在**给定参数点**上的误差，但不提供跨参数区域的保证。跨区域的保证需要额外的分布假设或 V&V 程序。

### 7.3.3 V&V 对两类模型的统一要求与适用边界

尽管机器学习与 ROM 的构造方法不同，可信性评估的逻辑结构一致：任何代理模型的精度声明，必须说明其在**哪个参数区域**、**相对于哪个参考**、**以何种范数**成立。这一要求可形式化为以下验证层级。

**代码验证层级。** 对 ROM，制造解方法与网格收敛检验确认降阶代码正确实现了投影或推断算法。对机器学习代理，代码验证的对应物是检查训练/验证流程是否存在数据泄漏、预处理是否正确，以及推断代码是否与训练时的一致性条件匹配。二者的共同目标是排除实现错误对误差的贡献。

**解验证层级。** 对 ROM，解验证估计离散化误差与降阶误差的相对大小。Surrogate-to-code 验证框架提供了一个可操作的范例：将代理模型的预测误差与 FOM 自身的离散化不确定度进行比较，若代理误差落在离散化“地板”以下，则代理的精度相对于参考模型而言不可区分。对机器学习代理，解验证要求报告模型误差相对于训练数据噪声水平或参考解数值精度的关系，而非仅报告测试集上的损失值。

**确认层级。** Validation 要求与**独立于构造过程**的参考对比。对 ROM 与机器学习代理而言，这意味着测试参数必须超出训练/快照采样范围。若独立参考不可得，则必须明确声明该代理的 Validation 尚未完成，其可信性仅限于训练覆盖范围内的 Verification 证据。这一要求与本章反复强调的原则一致：代码验证通过不蕴含模型验证通过，合成数据上的精度不构成现实有效性的证据。

上述共性框架的适用性受以下条件约束。第一，误差分解的表示误差项要求约化空间或假设类具有可计算的逼近性质；对深度神经网络，有限样本下的表示误差通常不可精确计算，只能以经验风险近似。第二，定理 7.3.4 的界逐点成立，但不提供参数空间中的一致界；跨参数区域的保证需要额外的光滑性假设或覆盖数分析。第三，V&V 层级中的“独立参考”在科学计算中通常由更高保真度的数值模拟提供，而非物理观测；这与 Validation 的严格定义存在差距，但为工程实践中的可信性排序提供了可操作的折中。第四，误差估计方法本身具有适用区域：DWR 方法在 ROM 外推区域有效，而机器学习误差模型在插值区域更可靠，选择误差估计器时需明确其训练或适用条件。

### 参考文献

投影型降阶与 Operator Inference 的表述见 Peherstorfer 与 Willcox 的 *Data-driven operator inference for nonintrusive projection-based model reduction*（*Computer Methods in Applied Mechanics and Engineering*, 306:196–215, 2016）及 Peherstorfer 的 *Sampling low-dimensional Markovian dynamics for preasymptotically recovering reduced models from data*（*SIAM Journal on Numerical Analysis*, 58(5):3003–3025, 2020）。DWR 与机器学习误差估计在 ROM 中的比较见 Blonigan 与 Parish 的 *Evaluation of dual-weighted residual and machine learning error estimation for projection-based reduced-order models of steady partial differential equations*（*Computer Methods in Applied Mechanics and Engineering*, 409:115988, 2023）。神经网络降阶模型的外推退化数据见《空气动力学学报》2024 年高速翼型流场预测研究第 3 节。Surrogate-to-code 验证框架见 Kamche 等 *Surrogate-to-code verification of a non-intrusive POD-GPR machine-learning emulator of peak thermomechanical fields*（arXiv:2607.16447, 2026）。V&V 的经典框架见 Oberkampf 与 Roy 的 *Verification and Validation in Scientific Computing*（Cambridge University Press, 2010）第 1–2 章。


### 7.3.4 实验设计：先预测，再运行

本节实验的设计原则是：全阶模型（FOM）必须具有已知的精确解或极高精度的参考解，使表示误差与拟合误差可分别计算；ROM 与机器学习代理必须在同一参数范围内构造，使二者的误差可直接比较；参数外推区必须显式分离，使分布外误差可独立报告。

**实验模型与数据生成。** 取一维参数化椭圆方程 $-\partial_{xx}u(x;\mu) = g(x;\mu)$，$x\in[0,1]$，边界条件 $u(0)=u(1)=0$。取 $\mu\in[0.1,1.0]$ 为参数，源项 $g(x;\mu)=\mu^2\sin(\pi x)+2\mu$。该问题的精确解为 $u(x;\mu)=\frac{\mu^2}{\pi^2}\sin(\pi x)+\mu x(1-x)$。FOM 取有限差分离散，网格 $N=200$，空间步长 $h=1/(N+1)$。由于精确解已知，FOM 的离散误差可按 $O(h^2)$ 理论阶计算，无需额外数值参考。

**两类代理模型的构造。** **ROM 分支**：在训练参数集 $\mu\in\{0.1,0.2,\dots,0.5\}$ 上计算 FOM 快照，使用主成分分析（POD）提取 $r=3$ 个约化基，并以 Galerkin 投影构造投影型 ROM。**机器学习分支**：在相同训练参数集与相同空间网格点上采集 FOM 快照，训练一个输入为 $(\mu,x)$、输出为 $u$ 的多层感知机，隐藏层宽度 $64$，激活函数 tanh，使用 Adam 优化器与固定随机种子。两分支使用完全相同的训练数据，以保证表示误差与拟合误差的比较条件一致。

**评价区间与划分。** 评价分三档。**分布内区** $\mu\in\{0.15,0.25,\dots,0.45\}$，位于训练参数范围内部。**边界邻近区** $\mu\in[0.55,0.65]$，紧邻训练范围之外。**远距外推区** $\mu\in[0.75,1.0]$，远离训练覆盖。每个参数取值下取 $N$ 个空间网格点计算误差。重复 $T=20$ 次独立训练（不同随机种子与快照采样），报告各区间相对 $L^2$ 误差的均值与标准差。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 回答：ROM 与 MLP 的误差分解中，表示误差与拟合误差各占多少；两者在分布内区、边界邻近区与远距外推区的误差如何变化。预测：分布内区 ROM 与 MLP 误差均接近 FOM 离散误差水平；边界邻近区误差上升，MLP 上升幅度可能大于 ROM（因神经网络在参数外推时缺乏显式基约束）；远距外推区两者误差均显著增大，且 ROM 的表示误差项（POD 截断）可能成为主导。 |
| 输入、参数、单位、数据来源与划分 | 参数 $\mu\in[0.1,1.0]$，无量纲；空间 $x\in[0,1]$，无量纲；源项 $g(x;\mu)=\mu^2\sin(\pi x)+2\mu$；FOM 网格 $N=200$；ROM 约化维度 $r=3$；训练参数 $\{0.1,\dots,0.5\}$，步长 $0.1$；分布内评价 $\mu\in\{0.15,\dots,0.45\}$；边界邻近 $\mu\in[0.55,0.65]$；远距外推 $\mu\in[0.75,1.0]$；重复 $T=20$；固定伪随机种子。 |
| 独立参考解/基线及选择理由 | 参考解为精确解 $u(x;\mu)=\frac{\mu^2}{\pi^2}\sin(\pi x)+\mu x(1-x)$，可精确计算任意参数与空间点上的真值，用于分离 FOM 离散误差与代理误差。基线为**零阶保持**代理：对任意 $\mu$，取最近的训练参数的 FOM 解作为预测。该基线不进行任何插值或外推，其误差指示“无代理能力”时的误差尺度。 |
| 误差指标、容差及其依据 | 相对 $L^2$ 误差 $\frac{\|u(\mu)-\hat{u}_r(\mu)\|_2}{\|u(\mu)\|_2}$，在 $N$ 个空间网格点上离散计算。容差取 $10^{-3}$，依据为 FOM 在 $N=200$ 下的理论离散误差 $O(h^2)\approx 2.5\times 10^{-5}$ 与训练数据的数值精度，用于区分“接近离散地板”与“显著超出”。报告 $T=20$ 次重复的均值与标准差。 |
| 环境、种子、设备与运行预算 | Python 3.11，NumPy 1.26，SciPy 1.11，PyTorch 2.2；种子 2026；单核 CPU；ROM 分支运行时间小于 1 分钟，MLP 分支运行时间约 5 分钟。 |

实验的变量关系可形式化如下。对固定参数 $\mu$，代理误差分解为
$$
e(\mu) \le e_{\text{repr}}(\mu) + e_{\text{fit}}(\mu),
$$
其中 $e_{\text{repr}}(\mu)=\inf_{v\in\mathcal{V}_r}\|u(\mu)-v\|$ 在 ROM 分支中由 POD 截断误差给出，在 MLP 分支中由网络表示能力隐含决定但不可直接计算；$e_{\text{fit}}(\mu)$ 在 ROM 分支中由 Galerkin 投影误差给出，在 MLP 分支中由训练损失与泛化误差共同决定。分布内区 $\mu\in[0.15,0.45]$ 的误差应接近 FOM 离散地板；边界邻近区与远距外推区的误差应显著超出该地板。若将相对误差对 $\mu$ 作图，预期曲线在内插区平坦、在训练边界 $\mu=0.5$ 附近出现拐点、在外推区单调上升。这一曲线形态即为 7.2 节所述“地图边界”在误差空间中的对应。

### 7.3.5 结果解释与本节结论

本节需报告的证据包括：FOM 网格规模与离散阶、ROM 约化维度、MLP 架构与训练设置、训练与评价参数区间、重复次数、各区间相对 $L^2$ 误差的均值与标准差。图中以底色区分分布内区、边界邻近区与远距外推区，以实线与虚线区分 ROM 与 MLP 误差曲线，纵轴为对数刻度的相对 $L^2$ 误差，横轴为参数 $\mu$。

**预期结果的分析框架。** 若实验按上述参数运行，各区间误差应呈现以下模式。第一，分布内区 $\mu\in[0.15,0.45]$ 的误差对 ROM 与 MLP 均应接近或略高于 FOM 离散地板 $10^{-4}$ 量级。第二，边界邻近区 $\mu\in[0.55,0.65]$ 的误差应上升一至两个数量级，且 MLP 的上升幅度可能大于 ROM，原因是 ROM 的约化基由训练参数的全局快照构造，具有一定的参数插值能力，而 MLP 在未见参数上的输出由网络权重的局部线性化行为决定，缺乏显式结构约束。第三，远距外推区 $\mu\in[0.75,1.0]$ 的误差对两者均应显著增大，但增大的机制不同：ROM 的误差可能主要由表示误差 $e_{\text{repr}}$ 主导（POD 基在远距参数下无法覆盖解流形的新方向），MLP 的误差则可能由拟合误差 $e_{\text{fit}}$ 与表示误差共同决定，且难以分离。

**本节结论。** 机器学习代理与 ROM 的误差在结构上共享同一分解：表示误差界定上限，拟合误差可通过数据与优化减小，二者共同决定给定参数点上的精度。定理 7.3.4 的形式化表述表明，该分解是三角不等式的直接推论，不依赖于模型的具体形式。V&V 层级的统一要求表明，任何代理模型的精度声明必须说明其在哪个参数区域、相对于哪个参考、以何种范数成立。本实验的设计将这一要求操作化：分布内区检验解验证，边界邻近区检验误差增长的起始位置，远距外推区检验误差估计方法本身的适用边界。实验的预期结果与 7.2 节的核心结论一致：代理模型的可信性在参数空间中不是均匀分布的，跨越训练边界后，误差的增长机制由表示误差主导，且不同模型的增长速率不可由单一指标预测。

**随堂提问：区分代理模型在“解验证”与“确认”两个层级上的证据。**

问题条件：给定一个由 POD-Galerkin 构造的 ROM，其训练参数为 $\mu\in[0.1,0.5]$。现有三组证据：（A）在 $\mu\in[0.1,0.5]$ 内，ROM 与 FOM 快照的相对误差为 $10^{-4}$；（B）在 $\mu\in[0.6,1.0]$ 内，ROM 与高保真 DNS 计算的相对误差为 $10^{-2}$；（C）在 $\mu\in[0.1,0.5]$ 内，FOM 自身的网格加密检验显示离散误差按 $O(h^2)$ 下降。请判断各组证据属于 Verification 的哪一层级或 Validation，并说明其可信性范围。

同学可能的回答与辨析。同学可能将（A）归为 Validation，理由是“误差很小”。辨析：（A）的比较对象是 FOM 快照，而 FOM 与 ROM 求解的是同一数学模型，故（A）属于解验证，不构成 Validation。（A）的可信性范围限于 $\mu\in[0.1,0.5]$ 的插值区间。对（B），同学可能认为高保真 DNS 与 ROM 的比较已足够构成 Validation。辨析：（B）确实是 Validation 的候选证据，但需检查 DNS 是否独立于 ROM 的构造过程、其自身不确定度是否被量化、以及 $\mu\in[0.6,1.0]$ 是否覆盖了目标用途的参数范围。若 DNS 本身未经验证，则（B）仅构成两个未验证模型之间的比较。对（C），同学可能忽略其重要性。辨析：（C）属于代码验证与解验证的混合：网格加密检验确认 FOM 实现正确且离散误差可控，是（A）与（B）中所有比较的前提；若 FOM 自身未经验证，则以其为参考的代理误差估计不可信。

本组最终解释：三组证据分别对应三个层级。（C）为代码验证与 FOM 解验证，构成后续比较的基础。（A）为代理模型的解验证，其可信性限于训练参数覆盖范围。（B）为 Validation 的候选证据，其可信性取决于 DNS 的独立性与不确定度量化，且其参数范围必须与目标用途匹配。三组证据不能相互替代：（A）不能升级为 Validation，（B）不能替代（A）在分布内的精细验证，（C）不能证明 ROM 或 DNS 适合目标用途。这一区分的实践意义在于：代理模型的可信性声明必须明确其证据属于哪一层级、覆盖哪个参数区域，否则“误差为 $10^{-4}$”或“与 DNS 一致”均不构成可检验的可信性陈述。

### 参考文献

投影型降阶与 Operator Inference 的表述见 Peherstorfer 与 Willcox 的 *Data-driven operator inference for nonintrusive projection-based model reduction*（*Computer Methods in Applied Mechanics and Engineering*, 306:196–215, 2016）及 Peherstorfer 的 *Sampling low-dimensional Markovian dynamics for preasymptotically recovering reduced models from data*（*SIAM Journal on Numerical Analysis*, 58(5):3003–3025, 2020）。DWR 与机器学习误差估计在 ROM 中的比较见 Blonigan 与 Parish 的 *Evaluation of dual-weighted residual and machine learning error estimation for projection-based reduced-order models of steady partial differential equations*（*Computer Methods in Applied Mechanics and Engineering*, 409:115988, 2023）。神经网络降阶模型的外推退化数据见《空气动力学学报》2024 年高速翼型流场预测研究第 3 节。Surrogate-to-code 验证框架见 Kamche 等 *Surrogate-to-code verification of a non-intrusive POD-GPR machine-learning emulator of peak thermomechanical fields*（arXiv:2607.16447, 2026）。V&V 的经典框架见 Oberkampf 与 Roy 的 *Verification and Validation in Scientific Computing*（Cambridge University Press, 2010）第 1–2 章。POD-Galerkin 约化基的表示误差分析见 Berkooz、Holmes 与 Lumley 的 *The proper orthogonal decomposition in the analysis of turbulent flows*（*Annual Review of Fluid Mechanics*, 25:539–575, 1993）。

## 7.4 AI关联：不确定性量化

7.1 节至 7.3 节分别讨论了统计泛化界、分布偏移下的风险关系以及数值离散误差的界。这三类分析均以“点预测”为对象，即关注预测值偏离真实值的期望程度。然而，在科学决策场景中，仅给出点预测及其期望误差界尚不足以支撑风险判断。决策者需要知道：给定置信水平 $1-\alpha$，真实值落在哪个区间内？该区间是否在独立数据上达到标称覆盖率？本节引入不确定性量化的基本框架，并聚焦于一种具有有限样本覆盖保证的构造方法。

### 7.4.1 基本定义

**定义 7.4.1（随机不确定性与认知不确定性）** 设数据生成过程为 $(X,Y)\sim P$。在给定输入 $X=x$ 的条件下，输出 $Y$ 的**随机不确定性**（aleatoric uncertainty）指由条件分布 $P(Y\mid X=x)$ 自身的离散程度所决定的不可约波动。即使拥有关于 $P$ 的完全知识，该波动仍然存在。**认知不确定性**（epistemic uncertainty）指由于对 $P$ 的知识不完整或模型参数估计不精确所引入的不确定性，原则上可通过增加信息或数据予以减小。

需要指出，这一区分在操作层面存在边界模糊性。同一不确定性来源在不同建模视角下可能被归入不同类别，且二者的可分离性依赖于具体的概率模型设定。本节后续的区间构造不依赖于对这一区分的精确量化，而直接以覆盖率为目标。

**定义 7.4.2（预测区间与覆盖率）** 设 $(X_1,Y_1),\dots,(X_n,Y_n)$ 与 $(X_{n+1},Y_{n+1})$ 为独立同分布样本。一个**预测区间**是一个集值映射 $C:\mathcal{X}\to 2^{\mathcal{Y}}$，其中 $2^{\mathcal{Y}}$ 表示 $\mathcal{Y}$ 的幂集。给定显著性水平 $\alpha\in(0,1)$，若
$$
\mathbb{P}\big(Y_{n+1}\in C(X_{n+1})\big)\ge 1-\alpha,
$$
其中概率同时取于训练样本与测试点，则称 $C$ 具有**边际覆盖率**至少 $1-\alpha$。上述概率不条件于 $X_{n+1}=x$，因此边际覆盖不保证对每个 $x$ 的条件覆盖率。**区间宽度** $\mathbb{E}[\text{Leb}(C(X))]$ 度量区间的信息含量，与覆盖率共同构成评价预测区间质量的两个维度。

**定义 7.4.3（非一致性分数）** 设 $\mathcal{A}$ 为在训练集上拟合的预测模型。**非一致性分数**是一个可测函数 $s:\mathcal{X}\times\mathcal{Y}\to\mathbb{R}$，用于度量候选标签 $y$ 与输入 $x$ 在模型 $\mathcal{A}$ 下的“不一致程度”。常用的选择包括绝对残差 $s(x,y)=|y-\hat{\mu}(x)|$ 与归一化残差 $s(x,y)=|y-\hat{\mu}(x)|/\hat{\sigma}(x)$。

### 7.4.2 分裂共形预测的覆盖保证

分裂共形预测（split conformal prediction）将数据划分为训练集与校准集，在校准集上计算非一致性分数的经验分位数，并以此构造预测区间。其有限样本覆盖保证不依赖于数据分布的具体形式，仅要求样本可交换性。

**定义 7.4.4（分裂共形预测区间）** 设训练集 $\mathcal{D}_{\text{train}}$ 用于拟合模型 $\hat{\mu}$，校准集 $\mathcal{D}_{\text{cal}}=\{(X_i,Y_i)\}_{i=1}^{m}$ 与测试点 $(X_{n+1},Y_{n+1})$ 均未参与训练。对每个校准样本计算 $S_i=s(X_i,Y_i)$，将其按升序排列为 $S_{(1)}\le\cdots\le S_{(m)}$。给定显著性水平 $\alpha\in(0,1)$，定义
$$
k_\alpha=\lceil (m+1)(1-\alpha)\rceil.
$$
若 $k_\alpha\le m$，令 $\hat{q}_{1-\alpha}=S_{(k_\alpha)}$；若 $k_\alpha=m+1$，令 $\hat{q}_{1-\alpha}=+\infty$。分裂共形预测区间定义为
$$
C_{\alpha}(x)=\{y\in\mathcal{Y}:s(x,y)\le\hat{q}_{1-\alpha}\}.
$$

**定理 7.4.5（分裂共形的有限样本覆盖保证）** 设 $(X_1,Y_1),\dots,(X_m,Y_m),(X_{n+1},Y_{n+1})$ 为可交换随机变量。令 $C_\alpha$ 由定义 7.14 构造。则
$$
\mathbb{P}\big(Y_{n+1}\in C_\alpha(X_{n+1})\big)\ge 1-\alpha.
$$
此外，若非一致性分数几乎必然互不相同，则
$$
\mathbb{P}\big(Y_{n+1}\in C_\alpha(X_{n+1})\big)\le 1-\alpha+\frac{1}{m+1}.
$$

**证明** 将测试点的非一致性分数 $S_{n+1}=s(X_{n+1},Y_{n+1})$ 与校准分数 $S_1,\dots,S_m$ 合并。由可交换性，$S_{n+1}$ 在 $m+1$ 个分数中的秩均匀分布于 $\{1,\dots,m+1\}$。事件 $Y_{n+1}\notin C_\alpha(X_{n+1})$ 等价于 $S_{n+1}>\hat{q}_{1-\alpha}$。由 $\hat{q}_{1-\alpha}$ 的定义，$S_{n+1}>\hat{q}_{1-\alpha}$ 意味着 $S_{n+1}$ 的秩至少为 $k_\alpha+1$。秩大于等于 $k_\alpha+1$ 的概率为
$$
\frac{m+1-k_\alpha}{m+1}.
$$
由于 $k_\alpha=\lceil (m+1)(1-\alpha)\rceil\ge (m+1)(1-\alpha)$，有
$$
\frac{m+1-k_\alpha}{m+1}\le \frac{m+1-(m+1)(1-\alpha)}{m+1}=\alpha.
$$
因此 $\mathbb{P}(Y_{n+1}\notin C_\alpha(X_{n+1}))\le\alpha$，即覆盖概率至少 $1-\alpha$。上界由秩的离散性直接得出。$\square$

该定理的核心特征在于：覆盖保证对任意样本量 $m$ 成立，不依赖渐近近似；不对数据分布 $P$ 做参数假设；不要求模型 $\hat{\mu}$ 正确指定。然而，覆盖是边际的而非条件的：对某些 $X$ 取值，实际覆盖率可能低于 $1-\alpha$。此外，保证依赖于可交换性假设。当测试样本与校准样本的分布不一致时，可交换性被破坏，覆盖保证不再成立。这一限制与 7.2 节讨论的分布外问题直接对应：共形预测在分布外输入上的覆盖率没有理论保证。

**推论 7.4.6（区间宽度的单调性）** 在定理 7.4.5 的条件下，对固定的校准集，$\hat{q}_{1-\alpha}$ 随 $\alpha$ 减小而单调不减。因此，覆盖率要求越高（$\alpha$ 越小），区间宽度 $\mathbb{E}[\text{Leb}(C_\alpha(X))]$ 在期望意义上越大。

**证明** 若 $\alpha_1<\alpha_2$，则 $(m+1)(1-\alpha_1)>(m+1)(1-\alpha_2)$，故 $k_{\alpha_1}\ge k_{\alpha_2}$。由 $S_{(k)}$ 的单调性，$\hat{q}_{1-\alpha_1}\ge\hat{q}_{1-\alpha_2}$。由于 $C_\alpha(x)=\{y:s(x,y)\le\hat{q}_{1-\alpha}\}$，阈值越大则区间包含的 $y$ 越多，故宽度不减。$\square$

推论 7.4.6 的形式化表述揭示了覆盖率与宽度之间的根本权衡：任何声称高覆盖率的预测区间，必须同时报告其宽度，否则覆盖率信息不具有可比性。

分裂共形预测的覆盖保证依赖于以下条件。第一，可交换性：校准样本与测试样本必须来自同一分布，且顺序不影响联合分布。在科学计算中，若测试任务的参数范围或初值条件超出校准集覆盖范围，可交换性不成立，定理 7.4.5 的结论失效。第二，非一致性分数的选择影响区间效率。以残差为分数时，区间宽度在异方差数据上可能过大；归一化分数可改善局部适应性，但需要可靠的尺度估计。第三，边际覆盖不蕴含条件覆盖。对特定的输入子群，覆盖率可能显著低于标称水平。Barber 等证明，在无额外假设的有限样本下，精确的条件覆盖不可能对任意分布实现。第四，共形预测仅校准点预测模型的不确定性表达，不改善模型本身的准确性；若基础模型在分布外区域系统性偏差，共形区间将以“覆盖真实值”为代价变得极宽或偏移。

### 参考文献

共形预测的奠基性工作见 Vovk、Gammerman 与 Shafer 的 *Algorithmic Learning in a Random World*（Springer, 2005；第 2 版 2022），该书系统建立了可交换性下的有效性理论。分裂共形预测的有限样本边际覆盖定理在文献中通常归于 Papadopoulos 等与 Vovk 的早期工作，其标准陈述见 Angelopoulos 与 Bates 的 *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*（arXiv:2107.07511, 2021）。条件覆盖的不可能性结果见 Barber、Candès、Ramdas 与 Tibshirani 的 *The limits of distribution-free conditional predictive inference*（*Information and Inference*, 10(2):455–482, 2021）以及同作者 *Predictive inference with the jackknife+*（*Annals of Statistics*, 49(1):486–507, 2021）。随机不确定性与认知不确定性的区分及其争议见 arXiv:2412.20892 与 arXiv:2505.22655 的讨论。

## 7.4.3 实验设计：先预测，再运行

本节实验的目标是将 7.4.2 节中分裂共形预测的有限样本覆盖保证置于可控的小规模回归问题中，观察覆盖率在独立测试集上的表现，并检验当测试分布偏离校准分布时，可交换性假设失效对覆盖率的实际影响。实验的设计原则是：基础预测模型与校准流程完全固定，使覆盖率的波动可归因于有限的校准样本量；同时显式构造一个参数外推的测试域，使域外覆盖率能够被独立记录。

**实验模型与数据生成。** 取一维输入 $x\in[0,4]$，真实目标函数 $f(x)=\sin(x)$。观测标签 $y=f(x)+\eta$，其中 $\eta\sim\mathcal{N}(0,\sigma^2)$，$\sigma=0.1$。基础预测模型 $\hat{\mu}$ 取三次多项式回归，参数通过普通最小二乘在训练集上估计。非一致性分数取绝对残差 $s(x,y)=|y-\hat{\mu}(x)|$。

**数据划分与域外构造。** 训练集取 $x\in[0,2]$ 上均匀采样的 $n_{\text{train}}=100$ 个点，用于拟合 $\hat{\mu}$。校准集取 $x\in[0,2]$ 上独立均匀采样的 $n_{\text{cal}}\in\{50, 200, 1000\}$ 个点，用于计算经验分位数 $\hat{q}_{1-\alpha}$。分布内测试集取 $x\in[0,2]$ 上独立均匀采样的 $n_{\text{test}}=2000$ 个点。分布外测试集取 $x\in[3,4]$ 上均匀采样的 $n_{\text{test}}=2000$ 个点，与训练及校准的输入支撑完全分离。标称覆盖率取 $1-\alpha\in\{0.80, 0.90, 0.95\}$。对每个 $(n_{\text{cal}}, \alpha)$ 组合，重复 $T=100$ 次独立采样，每次重新拟合模型、重新计算 $\hat{q}_{1-\alpha}$，并在同一组测试集上记录实测覆盖率与平均区间宽度。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | 回答：分裂共形预测在分布内测试集上的实测覆盖率是否达到标称覆盖率；有限校准样本量如何影响覆盖率的波动；在分布外测试集上覆盖率是否仍达到标称水平。预测：分布内覆盖率不低于标称值，且随 $n_{\text{cal}}$ 增大而更接近标称值，波动幅度约为 $1/\sqrt{n_{\text{cal}}}$ 量级；区间宽度随标称覆盖率提高而增大；分布外覆盖率显著低于标称值，且 $n_{\text{cal}}$ 的增大不能修复该偏差。 |
| 输入、参数、单位、数据来源与划分 | 输入 $x\in[0,4]$，无量纲；$f(x)=\sin(x)$；噪声 $\sigma=0.1$；训练集 $n_{\text{train}}=100$，$x\in[0,2]$；校准集 $n_{\text{cal}}\in\{50,200,1000\}$，$x\in[0,2]$；分布内测试集 $n_{\text{test}}=2000$，$x\in[0,2]$；分布外测试集 $n_{\text{test}}=2000$，$x\in[3,4]$；标称覆盖率 $1-\alpha\in\{0.80,0.90,0.95\}$；重复 $T=100$；伪随机种子固定。 |
| 独立参考解/基线及选择理由 | 参考解为 $f(x)=\sin(x)$ 本身，可精确计算每个测试点上的真值，从而判定是否被区间覆盖。基线为**固定宽度区间** $\hat{\mu}(x)\pm c$，其中 $c$ 取校准集上绝对残差的样本标准差 $1.96\hat{\sigma}$，对应正态近似下的 95\% 区间。该基线不依赖共形校准，用于对照共形方法在分布内是否获得与正态近似相当的宽度，在分布外是否同样失效。 |
| 误差指标、容差及其依据 | 指标为实测覆盖率 $\hat{c}=\frac{1}{n_{\text{test}}}\sum_{i}\mathbb{1}\{y_i\in C_\alpha(x_i)\}$ 与平均区间宽度 $\bar{w}=\frac{1}{n_{\text{test}}}\sum_{i}|C_\alpha(x_i)|$。覆盖率的抽样标准误在 $n_{\text{test}}=2000$ 下约为 $\sqrt{\alpha(1-\alpha)/n_{\text{test}}}\le 0.011$。容差取 $2$ 倍标准误，即 $0.022$，用于判定实测覆盖率是否显著偏离标称值。$T=100$ 次重复用于报告覆盖率与宽度的分布，而非仅报告单次结果。 |
| 环境、种子、设备与运行预算 | Python 3.11，NumPy 1.26，scikit-learn 1.4；种子 2026；单核 CPU；预计运行时间小于 3 分钟。 |

实验的变量关系可形式化如下。对固定校准集与测试点，覆盖事件 $\{Y\in C_\alpha(X)\}$ 的概率在可交换性下满足定理 7.5 的界 $1-\alpha\le\mathbb{P}(Y\in C_\alpha(X))\le 1-\alpha+1/(n_{\text{cal}}+1)$。因此，实测覆盖率 $\hat{c}$ 在重复试验中的期望应落在该区间内，其波动由 $n_{\text{test}}$ 与 $n_{\text{cal}}$ 共同决定。在分布外测试集上，可交换性被破坏，定理 7.5 的前提不成立，实测覆盖率无理论下界。若将 $\hat{c}$ 对 $n_{\text{cal}}$ 作图，分布内曲线应收敛于标称值附近，分布外曲线则可能停留在任意水平，且不随 $n_{\text{cal}}$ 增大而改善。若将 $\bar{w}$ 对 $1-\alpha$ 作图，依据推论 7.4.6，宽度应随标称覆盖率提高而单调增大。

### 7.4.4 结果解释与本节结论

本节需报告的证据包括：训练集、校准集与两类测试集的样本量；标称覆盖率 $1-\alpha$ 的三个取值；重复次数 $T=100$；分布内与分布外实测覆盖率的均值、标准差与 $95\%$ 经验分位区间；平均区间宽度的均值与标准差。图中以横轴为标称覆盖率、纵轴为实测覆盖率绘制对角线参考线，以实线与虚线区分分布内与分布外结果，另以误差棒表示重复试验的标准差；宽度图以相同横轴、纵轴为平均区间宽度。

**预期结果的分析框架。** 若实验按上述参数运行，分布内覆盖率应呈现以下模式。第一，实测覆盖率 $\hat{c}$ 的均值应不低于标称值 $1-\alpha$，且通常略高于标称值，高出的幅度不超过 $1/(n_{\text{cal}}+1)$。当 $n_{\text{cal}}=50$ 时，该上界约为 $0.02$；当 $n_{\text{cal}}=1000$ 时，该上界约为 $0.001$。第二，$\hat{c}$ 在 $T=100$ 次重复中的标准差应随 $n_{\text{cal}}$ 增大而减小，主要来源是 $\hat{q}_{1-\alpha}$ 的抽样波动。第三，平均区间宽度应随标称覆盖率提高而增大，且在 $n_{\text{cal}}$ 较小时宽度更大，因为经验分位数在样本较少时更保守。这些模式与定理 7.5 及推论 7.2 的结论一致。

**本节结论。** 分裂共形预测在可交换性成立时提供有限样本的边际覆盖保证，其覆盖率不低于标称值，且超出幅度不超过 $1/(n_{\text{cal}}+1)$。实验设计将这一保证操作化：分布内测试验证覆盖率的有效性，分布外测试检验可交换性失效后的覆盖率退化，基线对比检验共形方法相对于参数化区间的宽度效率。实验的预期结果与 7.2 节及 7.3 节的核心结论一致：任何覆盖保证都以分布假设为前提，分布改变后覆盖率无理论下界，增大校准样本量不能修复分布偏移带来的偏差。因此，报告预测区间时必须同时报告标称覆盖率、实测覆盖率、区间宽度与评价分布，四者缺一不可。

**待核验的风险：把模型输出的置信数值直接解释为真实正确率，或假设域内校准在域外自动成立。**

第一种表现：模型对某个输入输出“置信度 $0.9$”，报告者据此声称“该预测有 $90\%$ 的概率正确”。这一解释混淆了两个不同的概率对象。置信数值通常是模型在训练分布上的经验频率或 softmax 输出的归一化值，它不构成对当前输入的条件正确率估计。共形预测的覆盖率是边际的：$\mathbb{P}(Y\in C_\alpha(X))\ge 1-\alpha$ 中的概率同时取于校准样本与测试点，不条件于 $X=x$。即使边际覆盖率达到 $90\%$，对某些 $x$ 的条件覆盖率可能显著低于 $90\%$。Barber 等证明，在无额外假设的有限样本下，精确的条件覆盖不可能对任意分布实现。因此，将边际覆盖率解读为逐样本正确率，在数学上不成立。

第二种表现：校准在分布内达到标称覆盖率，报告者据此声称“区间在部署中也有效”。定理 7.4.5 的证明依赖可交换性：校准样本与测试样本的联合分布对顺序不变。当测试输入落在校准支撑之外时，可交换性被破坏，定理 7.4.5 的证明步骤不成立，覆盖率无下界。这一失效不是有限样本波动，不能通过增大 $n_{\text{cal}}$ 修复。7.4.3 节的分布外测试集即用于显式检验这一点。

形式化回应如下。设校准分布为 $P_{\text{cal}}$，测试分布为 $P_{\text{test}}$。若 $P_{\text{cal}}\ne P_{\text{test}}$，则覆盖概率
$$
\mathbb{P}_{(X,Y)\sim P_{\text{test}}}\big(Y\in C_\alpha(X)\big)
$$
不再由定理 7.4.5 控制。其数值可写为
$$
\mathbb{E}_{P_{\text{test}}}\big[\mathbb{1}\{s(X,Y)\le\hat{q}_{1-\alpha}\}\big],
$$
其中 $\hat{q}_{1-\alpha}$ 由 $P_{\text{cal}}$ 上的分数分位数决定。若 $P_{\text{test}}$ 下 $s(X,Y)$ 的分布整体右移（外推误差增大），则该期望可任意小。这一推导说明：共形预测的有效性是相对于校准分布的局部性质，不是模型的全局属性。

本节的讨论提示旨在训练读者在阅读“置信度”或“覆盖率”报告时首先追问其概率对象与分布条件。上述两种误读是方法论层面的结构性风险，不代表任何具体已部署系统均存在此问题；其作用是提供一种检查标准：任何覆盖率声明必须附带校准分布、测试分布、样本量与边际/条件覆盖的区分，四者缺一不可。

**随堂提问：设计两个模型覆盖率相近但区间宽度不同的比较题。**

问题条件：给定两个分裂共形预测模型 A 与 B，校准集大小均为 $n_{\text{cal}}=500$，标称覆盖率均为 $90\%$，在同一分布内测试集上实测覆盖率分别为 $0.901$ 与 $0.903$，平均区间宽度分别为 $0.8$ 与 $0.3$。两个模型使用相同的非一致性分数定义，但基础预测模型不同：A 使用三次多项式回归，B 使用五次多项式回归。请判断哪个模型的预测区间更可取，并说明理由。

同学可能的回答与辨析。第一种回答：A 的覆盖率略低，因此 B 更可取。辨析：覆盖率差异 $0.002$ 远小于 $n_{\text{test}}=2000$ 下的抽样标准误 $0.011$，不构成统计显著差异；覆盖率相近时，宽度更小的区间信息量更大，因此 B 更可取，但理由应是宽度而非覆盖率。第二种回答：B 的宽度更小，说明 B 的模型更准确，因此 B 一定更可取。辨析：宽度更小确实说明 B 的区间信息量更大，但需检查 B 的宽度优势是否在分布外仍然保持。五次多项式在外推区通常比三次多项式更不稳定，若部署场景包含外推输入，B 的分布外覆盖率可能显著低于 A，此时 A 的较宽区间反而更稳健。第三种回答：两个模型覆盖率相同，无法比较。辨析：覆盖率与宽度是两个独立维度，覆盖率相近时宽度决定信息量，但宽度优势必须与分布外行为、模型复杂度及计算代价共同评估。

本组最终解释：在分布内且覆盖率统计上无显著差异时，B 的区间宽度更小，信息量更高，若部署场景限于校准分布覆盖范围，B 更可取。但这一结论不能外推至分布外：五次多项式的外推不稳定性可能使 B 在域外覆盖率显著下降，此时需报告分布外覆盖率与宽度，并按部署场景的输入分布选择模型。比较两个预测区间的正确方式是同时报告覆盖率、宽度、校准分布与测试分布，而非仅比较单一指标。覆盖率相近但宽度不同的情形恰说明：覆盖率是预测区间的必要条件而非充分条件，宽度与分布条件共同决定区间的实际可信性。

### 参考文献

共形预测的奠基性工作见 Vovk、Gammerman 与 Shafer 的 *Algorithmic Learning in a Random World*（Springer, 2005；第 2 版 2022）。分裂共形预测的有限样本覆盖定理的标准陈述见 Angelopoulos 与 Bates 的 *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*（arXiv:2107.07511, 2021）。条件覆盖的不可能性结果见 Barber、Candès、Ramdas 与 Tibshirani 的 *The limits of distribution-free conditional predictive inference*（*Information and Inference*, 10(2):455–482, 2021）以及同作者 *Predictive inference with the jackknife+*（*Annals of Statistics*, 49(1):486–507, 2021）。分布偏移下共形预测的失效分析见 Tibshirani 等 *Conformal Prediction Under Covariate Shift*（*Advances in Neural Information Processing Systems*, 32, 2019）。随机不确定性与认知不确定性的区分及其操作争议见 arXiv:2412.20892 与 arXiv:2505.22655 的讨论。